# NB 01 — Extracción de Facturas
**Proyecto 5 · Automatización Contable**

**Input:** PDFs e imágenes en `data/input/compras/` y `data/input/ventas/`  
**Output:** JSON estructurado por factura en `data/processed/`

**Empresa cliente:** CT PRIME CONSULTING SAC · RUC 20563642930

## 0. Dependencias

In [1]:
import anthropic
import base64
import json
import fitz  # pymupdf
import io
import os
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

# Cargar API key desde .env compartido
ENV_PATH = Path('D:/Proyecto_Gabriel/02_Agente_IA/Skill_financiero/.env')
load_dotenv(dotenv_path=ENV_PATH)

BASE_DIR      = Path('../')
INPUT_COMPRAS = BASE_DIR / 'data/input/compras'
INPUT_VENTAS  = BASE_DIR / 'data/input/ventas'
OUTPUT_DIR    = BASE_DIR / 'data/processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUC_EMPRESA = '20563642930'

client = anthropic.Anthropic()
api_ok = 'SI' if os.environ.get('ANTHROPIC_API_KEY') else 'NO ENCONTRADA'
print(f'Setup OK - PyMuPDF {fitz.version[0]}')
print(f'API key cargada: {api_ok}')


Setup OK - PyMuPDF 1.27.2.3
API key cargada: SI


## 0b. Copia inicial de facturas de referencia
Ejecutar solo la primera vez para copiar las facturas desde `Proyecto_contable/`.
En producción, Carlos coloca los PDFs directamente en `data/input/compras/` o `data/input/ventas/`.

In [2]:
import shutil

# Demo: las facturas se colocan directamente en data/input/compras o data/input/ventas
# Esta celda de copia automática queda desactivada (no hay carpeta de referencia para la demo)
SRC_COMPRAS = BASE_DIR / 'data/input_referencia/compras'
SRC_VENTAS  = BASE_DIR / 'data/input_referencia/ventas'

def copiar_facturas_referencia():
    copiadas = 0
    for src, dst in [(SRC_COMPRAS, INPUT_COMPRAS), (SRC_VENTAS, INPUT_VENTAS)]:
        if not src.exists():
            print(f'  Carpeta no encontrada: {src}')
            continue
        for archivo in src.iterdir():
            if archivo.suffix.lower() in {'.pdf', '.jpg', '.jpeg', '.png'}:
                dest = dst / archivo.name
                if not dest.exists():  # no sobreescribir si ya fue copiado
                    shutil.copy2(archivo, dest)
                    copiadas += 1
    print(f'Facturas copiadas: {copiadas}')
    print(f'  Compras: {len(list(INPUT_COMPRAS.iterdir()))} archivos')
    print(f'  Ventas:  {len(list(INPUT_VENTAS.iterdir()))} archivos')

copiar_facturas_referencia()

  Carpeta no encontrada: ..\data\input_referencia\compras
  Carpeta no encontrada: ..\data\input_referencia\ventas
Facturas copiadas: 0
  Compras: 0 archivos
  Ventas:  63 archivos


## 1. Funciones de Conversión (PDF / Imagen → base64)

In [3]:
def archivo_a_base64(ruta: Path) -> tuple[str, str]:
    """
    Convierte PDF o imagen a base64 usando PyMuPDF (no requiere Poppler).
    Para PDFs renderiza la primera página a JPEG 200 DPI.
    """
    ext = ruta.suffix.lower()

    if ext == '.pdf':
        doc = fitz.open(str(ruta))
        page = doc[0]  # primera página
        mat = fitz.Matrix(200/72, 200/72)  # 200 DPI
        pix = page.get_pixmap(matrix=mat, alpha=False)
        img_bytes = pix.tobytes('jpeg')
        doc.close()
        return base64.b64encode(img_bytes).decode(), 'image/jpeg'

    elif ext in ['.jpg', '.jpeg']:
        with open(ruta, 'rb') as f:
            return base64.b64encode(f.read()).decode(), 'image/jpeg'

    elif ext == '.png':
        with open(ruta, 'rb') as f:
            return base64.b64encode(f.read()).decode(), 'image/png'

    else:
        raise ValueError(f'Formato no soportado: {ext}')

## 2. Extracción con Claude API (visión)

In [4]:
PROMPT_EXTRACCION = """
Eres un asistente especializado en documentos tributarios peruanos.
Extrae TODOS los campos del comprobante de pago en la imagen.

PASO 1 — Antes de nada, escribe UNA línea con el texto EXACTO de serie y número
del COMPROBANTE DE PAGO mismo (no de otras referencias que aparezcan en el documento).
El serie-número real del comprobante aparece SIEMPRE inmediatamente debajo o junto a la
etiqueta del tipo de documento ("FACTURA ELECTRÓNICA", "BOLETA DE VENTA ELECTRÓNICA",
"NOTA DE CRÉDITO ELECTRÓNICA", etc.), en formato "SERIE-NUMERO" (ej. "F002-00005396",
"E001-2"). IGNORA cualquier otro número con formato similar que aparezca en otra parte
del documento con una etiqueta distinta — por ejemplo "N° DE COTIZACIÓN", "N° DE PEDIDO",
"N° DE ORDEN", "N° DE GUÍA" u otras referencias internas del comercio NO son el serie-número
del comprobante, aunque tengan un formato parecido (letra-numero). Formato de esta línea:
SERIE_NUMERO_DETECTADO: <texto exacto>

PASO 2 — Luego, en una nueva línea, devuelve ÚNICAMENTE un JSON con esta estructura exacta:

{
  "tipo_doc": "FACTURA | BOLETA | NOTA_CREDITO | NOTA_DEBITO | RECIBO",
  "codigo_tipo_doc": "01 | 03 | 07 | 08 | 02 | 04",
  "serie": "F001",
  "numero": "00012345",
  "fecha_emision": "YYYY-MM-DD",
  "fecha_vencimiento_pago": "YYYY-MM-DD o null",
  "ruc_emisor": "20XXXXXXXXX",
  "razon_social_emisor": "Nombre legal del emisor",
  "ruc_receptor": "20XXXXXXXXX o null",
  "razon_social_receptor": "Nombre legal del receptor o null",
  "tipo_doc_identidad_receptor": "1 | 4 | 6 | 7 | 0",
  "numero_doc_identidad_receptor": "numero de DNI/RUC/pasaporte del receptor, o null",
  "descripcion_servicio": "Descripción principal del bien/servicio",
  "base_imponible": 0.00,
  "igv": 0.00,
  "total": 0.00,
  "moneda": "PEN | USD",
  "tipo_cambio": 0.000,
  "tiene_detraccion": false,
  "monto_detraccion": 0.00,
  "doc_referencia": "E001-00000030 o null",
  "otorga_credito_igv": true,
  "deducible_renta": true,
  "confianza_extraccion": 0.95
}

Reglas:
- codigo_tipo_doc: 01=Factura, 03=Boleta, 07=Nota de Crédito, 08=Nota de Débito, 02=Recibo por Honorarios, 04=Liquidación de Compra
- tipo_doc / codigo_tipo_doc: usa el rotulo LITERAL impreso en la parte superior del comprobante (ej. "FACTURA ELECTRONICA" = 01/FACTURA, "BOLETA DE VENTA ELECTRONICA" = 03/BOLETA, "NOTA DE CREDITO ELECTRONICA" = 07/NOTA_CREDITO). No infieras el tipo de documento a partir de otros campos ni asumas boleta por defecto si el rotulo dice factura.
- IMPORTANTE — identificación de EMISOR vs RECEPTOR: el EMISOR es la persona o empresa cuyo nombre y RUC aparecen en el encabezado/caja superior del comprobante, junto al número de serie-correlativo (ej. "E001-2"). El RECEPTOR es quien aparece después de la etiqueta "Señor(es):", "Cliente:" o "Razón Social del Adquirente". Guíate SIEMPRE por la posición y las etiquetas del documento — NUNCA asumas que el emisor es una empresa solo porque su nombre "suena" a empresa. Una persona natural (con RUC propio) puede emitir facturas electrónicas a una empresa, por ejemplo por alquiler de bienes o servicios personales; en ese caso la persona natural ES el emisor y la empresa ES el receptor.
- "numero": debe ser EXACTAMENTE el número que escribiste en SERIE_NUMERO_DETECTADO (la parte después del guion), rellenado con ceros a la izquierda si quieres, pero el valor numérico debe coincidir. Ej. si SERIE_NUMERO_DETECTADO fue "E001-2", numero debe ser "2" o "00000002" — NUNCA "00000001" ni ningún otro valor distinto al que detectaste en el paso 1.
- razon_social_emisor y razon_social_receptor: deben ser la denominación o razón social LEGAL, o el nombre completo si es persona natural (ej: "MEDIA SOLUTION E.I.R.L.", "SODIMAC PERU S.A.", "CORNEJO RETO JOSE ROBERTO ANDRE"). NUNCA una dirección, domicilio fiscal, ciudad ni referencia geográfica. Si el documento muestra el domicilio debajo del nombre, extrae SOLO el nombre legal o el nombre de la persona.
- Para NC/ND: "numero" debe ser el número PROPIO de la NC/ND (ej: para "NC E001-03", numero="00000003"), NO el número del comprobante que modifica. El comprobante modificado va ÚNICAMENTE en "doc_referencia".
- doc_referencia: para tipo 07 (NC) y 08 (ND), extraer la serie-número del comprobante que modifica (ej: "E001-00000030"). Para otros tipos, null.
- otorga_credito_igv: true si codigo_tipo_doc es 01 o 04 y el IGV está discriminado. false para 03 (Boleta) y 02 (Honorarios).
- deducible_renta: true para 01 (Factura), 02 (Honorarios) y 04 (Liquidación Compra). false para 03 (Boleta) por regla general.
- Para NC/ND (07/08): otorga_credito_igv y deducible_renta heredan del comprobante que modifican — dejar true por defecto.
- Si base_imponible no aparece explícita, calcúlala: total / 1.18
- Si IGV no aparece, calcúlalo: base_imponible * 0.18
- fecha_vencimiento_pago: solo si el comprobante muestra explícitamente una fecha de vencimiento distinta a la de emisión; si no aparece, usa null.
- tipo_doc_identidad_receptor: 1=DNI, 4=Carnet de Extranjería, 6=RUC, 7=Pasaporte, 0=otro/no identificado. Para FACTURA siempre es 6. Para BOLETA, usa el documento que aparezca (DNI es lo más común); si no hay documento de identidad visible, usa "0".
- numero_doc_identidad_receptor: el número del documento indicado en tipo_doc_identidad_receptor. Si es RUC, debe coincidir con ruc_receptor. Si tipo_doc_identidad_receptor es 6, este campo NUNCA debe quedar null — captura el RUC del receptor tal como aparece.
- tipo_cambio: solo si moneda es "USD" y el comprobante muestra el tipo de cambio explícitamente; si no aparece o moneda es "PEN", usa null.
- confianza_extraccion: 1.0 = todos los campos legibles, 0.7 = imagen borrosa o campos incompletos
- No uses bloques de markdown (```); despues de la linea SERIE_NUMERO_DETECTADO, escribe el JSON en texto plano
"""

def extraer_factura(ruta: Path, tipo_operacion: str) -> dict:
    """
    tipo_operacion: 'COMPRA' o 'VENTA'
    Retorna dict con todos los campos extraídos + metadatos.
    """
    b64, media_type = archivo_a_base64(ruta)
    
    response = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=1024,
        messages=[{
            'role': 'user',
            'content': [
                {
                    'type': 'image',
                    'source': {'type': 'base64', 'media_type': media_type, 'data': b64}
                },
                {'type': 'text', 'text': PROMPT_EXTRACCION}
            ]
        }]
    )
    
    texto = response.content[0].text.strip()
    # Limpiar markdown si Claude lo devuelve con bloques ```json
    if '```' in texto:
        partes = texto.split('```')
        texto = partes[1][4:] if partes[1].startswith('json') else partes[1]
    # Descartar la linea SERIE_NUMERO_DETECTADO y cualquier texto antes del JSON
    inicio_json = texto.find('{')
    if inicio_json > 0:
        texto = texto[inicio_json:]

    datos = json.loads(texto)
    datos['tipo_operacion'] = tipo_operacion
    datos['archivo_origen'] = ruta.name
    datos['procesado_en'] = datetime.now().isoformat()
    
    return datos

## 3. Procesamiento en lote

In [5]:
FORMATOS_VALIDOS = {'.pdf', '.jpg', '.jpeg', '.png'}

def procesar_carpeta(carpeta: Path, tipo_operacion: str) -> list[dict]:
    archivos = [f for f in carpeta.iterdir() if f.suffix.lower() in FORMATOS_VALIDOS]
    resultados = []
    errores = []
    
    print(f'\n=== {tipo_operacion}: {len(archivos)} archivos ===')
    
    for archivo in sorted(archivos):
        print(f'  Procesando: {archivo.name}... ', end='')
        try:
            datos = extraer_factura(archivo, tipo_operacion)
            resultados.append(datos)
            print(f'OK (confianza: {datos.get("confianza_extraccion", "?")})')
        except Exception as e:
            print(f'ERROR: {e}')
            errores.append({'archivo': archivo.name, 'error': str(e)})
    
    if errores:
        print(f'\nErrores ({len(errores)}):')
        for e in errores:
            print(f'  - {e["archivo"]}: {e["error"]}')
    
    return resultados

In [6]:
# Ejecutar extracción
facturas_compras = procesar_carpeta(INPUT_COMPRAS, 'COMPRA')
facturas_ventas  = procesar_carpeta(INPUT_VENTAS,  'VENTA')

todas_las_facturas = facturas_compras + facturas_ventas

# Deduplicación dentro del lote por (ruc_emisor, serie, numero, total, fecha_emision)
# Se exige que coincidan TAMBIEN monto y fecha -- si el numero se lee mal por
# la extraccion, un monto o fecha distintos evitan descartar una factura real.
vistas = set()
facturas_unicas = []
for f in todas_las_facturas:
    clave = (
        str(f.get('ruc_emisor', '')),
        str(f.get('serie', '')),
        str(f.get('numero', '')),
        str(round(float(f.get('total') or 0), 2)),
        str(f.get('fecha_emision', '')),
    )
    if clave not in vistas:
        vistas.add(clave)
        facturas_unicas.append(f)
    else:
        print(f'  ⚠️ Duplicado en lote: RUC={clave[0]} {clave[1]}-{clave[2]} total={clave[3]} fecha={clave[4]} — descartado')
todas_las_facturas = facturas_unicas

n_compras = sum(1 for f in todas_las_facturas if f.get('tipo_operacion') == 'COMPRA')
n_ventas  = sum(1 for f in todas_las_facturas if f.get('tipo_operacion') == 'VENTA')
print(f'\nTotal único: {len(todas_las_facturas)} ({n_compras} compras, {n_ventas} ventas)')


=== COMPRA: 0 archivos ===

=== VENTA: 63 archivos ===
  Procesando: 20601509157 E001-2421.pdf... 

OK (confianza: 0.92)
  Procesando: E001-2248.pdf... 

OK (confianza: 0.95)
  Procesando: E001-2250.pdf... 

OK (confianza: 0.95)
  Procesando: E001-2276.pdf... 

OK (confianza: 0.92)
  Procesando: E001-2278.pdf... 

...

OK (confianza: 0.92)
  Procesando: PDF-DOC-E001-30320607880736.pdf... 

OK (confianza: 0.92)
  Procesando: PDF-DOC-E001-30420607880736.pdf... 

OK (confianza: 0.85)
  Procesando: PDF-DOC-E001-520610992782.pdf... 

OK (confianza: 0.95)
  Procesando: PDF-DOC-E001-9910257456574.pdf... 

OK (confianza: 0.92)
  ⚠️ Duplicado en lote: RUC=20304177552 E001-00000966 total=149860.0 fecha=2024-07-17 — descartado

Total único: 62 (0 compras, 62 ventas)


## 4. Guardar resultados

In [7]:
# Guardar JSON individual por factura
for factura in todas_las_facturas:
    nombre_base = Path(factura['archivo_origen']).stem
    salida = OUTPUT_DIR / f'{nombre_base}_extraido.json'
    with open(salida, 'w', encoding='utf-8') as f:
        json.dump(factura, f, ensure_ascii=False, indent=2)

# Guardar resumen consolidado
resumen_path = OUTPUT_DIR / 'facturas_extraidas.json'
with open(resumen_path, 'w', encoding='utf-8') as f:
    json.dump(todas_las_facturas, f, ensure_ascii=False, indent=2)

print(f'Archivos guardados en: {OUTPUT_DIR}')
print(f'Resumen consolidado: {resumen_path}')

Archivos guardados en: ..\data\processed
Resumen consolidado: ..\data\processed\facturas_extraidas.json


## 5. Revisión de resultados

In [8]:
import pandas as pd

df = pd.DataFrame(todas_las_facturas)
cols_display = ['tipo_operacion', 'tipo_doc', 'serie', 'numero', 'fecha_emision',
                'razon_social_emisor', 'base_imponible', 'igv', 'total', 'confianza_extraccion']
cols_disponibles = [c for c in cols_display if c in df.columns]

print('=== FACTURAS EXTRAÍDAS ===')
print(df[cols_disponibles].to_string(index=False))

print(f'\n=== CONFIANZA BAJA (< 0.80) — revisar manualmente ===')
baja_confianza = df[df['confianza_extraccion'] < 0.80] if 'confianza_extraccion' in df.columns else pd.DataFrame()
if len(baja_confianza) > 0:
    print(baja_confianza[['archivo_origen', 'confianza_extraccion']].to_string(index=False))
else:
    print('  Ninguna — todas las facturas con confianza >= 0.80')

=== FACTURAS EXTRAÍDAS ===
tipo_operacion tipo_doc serie   numero fecha_emision                                           razon_social_emisor  base_imponible      igv     total  confianza_extraccion
         VENTA  FACTURA  E001     2421    2024-06-18          OCEANO AZUL SOLUCIONES INTEGRALES S.A.C. OASI S.A.C.       131567.29 23662.71 155449.00                  0.92
         VENTA  FACTURA  E001     2248    2024-03-07          OCEANO AZUL SOLUCIONES INTEGRALES S.A.C. OASI S.A.C.        16935.00  3048.30  19983.30                  0.95
         VENTA  FACTURA  E001     2250    2024-11-03          OCEANO AZUL SOLUCIONES INTEGRALES S.A.C. OASI S.A.C.       392742.00 70693.56 463435.56                  0.95
         VENTA  FACTURA  E001     2276    2024-01-04          OCEANO AZUL SOLUCIONES INTEGRALES S.A.C. OASI S.A.C.         6305.08  1134.92   8774.48                  0.92
         VENTA  FACTURA  E001     2278    2024-01-04          OCEANO AZUL SOLUCIONES INTEGRALES S.A.C. OASI S.A.C